# Week 6 · Extra projects

Optional EDA (Exploratory Data Analysis) practice beyond the lab: three small CSVs, the same report in three fields (IT, AI / Data Science, Cyber Security). Only `pandas` and `matplotlib.pyplot`, and only tools from this week: `read_csv`, new columns, boolean filtering, `groupby`, `sort_values`, `apply` + `lambda`, `value_counts`, a bar chart and `to_csv`.

Each script reads its CSV by its bare file name, so put the CSV in the **same folder** as the `.py` file and run it from there.

## Server Fleet Report (IT)

**Data:** `01 - Series and DataFrame/examples/servers.csv` (copy it into your working folder, or point `pd.read_csv()` at its path): 10 servers, 3 regions, the same file every topic this week already used.

1. **Load and look.** `pd.read_csv()` it. Print `.head()`, `.dtypes`, and `len()`. How many servers, how many columns, what type is each?
2. **Add a column.** `percent = (used / total) * 100`: no loop, the whole-column arithmetic from Topic 1.
3. **Ask a question with a filter.** How many servers are at or above 90%? Use boolean filtering (Topic 2), not a loop.
4. **Ask a question with a group.** What's the average `percent`, per region? `.groupby("region")["percent"].mean()`.
5. **Sort it.** Which single server is most at risk? `.sort_values()`, descending, take the top row.
6. **Classify it.** Add a `status` column using `.apply()` with a lambda: `"OVER LIMIT"` / `"WARNING"` / `"OK"`, same thresholds as the walkthrough.
7. **Chart it.** A bar chart of servers per region, or per status: `.value_counts().plot(kind="bar")`, `plt.savefig(...)`.
8. **Save it.** Write just the `OVER LIMIT` rows to a new CSV, `servers_at_risk.csv`, with `.to_csv(index=False)`.
9. **Write two sentences.** In a comment at the top of your script: what did you actually find? (e.g. "west has the highest average usage; srv-03 is the single most at-risk server.") An EDA that doesn't end in a finding isn't finished.

In [ ]:
# FINDINGS (two sentences):
# north has the highest average usage (88.7%), ahead of west (81.0%) and east (71.8%),
# and 5 of the 10 servers are at or above 90%.
# srv-03 (west) is the single most at-risk server at 120%, and 3 servers are already OVER LIMIT.

import pandas as pd
import matplotlib.pyplot as plt

# 1. Load and look
df = pd.read_csv("servers.csv")
print(df.head())
print()
print(df.dtypes)
print()
print(f"Servers (rows): {len(df)}")
print(f"Columns: {len(df.columns)} -> {df.columns.tolist()}")

# 2. Add a column - whole-column arithmetic, no loop
df["percent"] = (df["used"] / df["total"]) * 100

# 3. A question with a filter: how many servers are at or above 90%?
high_usage = df[df["percent"] >= 90]
print(f"\nServers at or above 90%: {len(high_usage)}")
print(high_usage[["hostname", "region", "percent"]])

# 4. A question with a group: average percent per region
region_average = df.groupby("region")["percent"].mean().round(1)
print("\nAverage percent per region (highest first):")
print(region_average.sort_values(ascending=False))

# 5. Sort it: the single most at-risk server
ranked = df.sort_values("percent", ascending=False)
most_at_risk = ranked.iloc[0]
print(f"\nMost at risk: {most_at_risk['hostname']} ({most_at_risk['region']}) "
      f"at {most_at_risk['percent']:.0f}%")

# 6. Classify it: status column with apply + lambda (same thresholds as the walkthrough)
df["status"] = df["percent"].apply(
    lambda p: "OVER LIMIT" if p >= 100 else "WARNING" if p >= 90 else "OK"
)
print()
print(df)
print()
print(df["status"].value_counts())

# 7. Chart it: servers per status
status_counts = df["status"].value_counts()
status_counts.plot(kind="bar")
plt.title("Servers per status")
plt.xlabel("Status")
plt.ylabel("Number of servers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("servers_per_status.png")
print("\nSaved servers_per_status.png")
plt.show()

# 8. Save it: only the OVER LIMIT rows
at_risk = df[df["status"] == "OVER LIMIT"]
at_risk.to_csv("servers_at_risk.csv", index=False)
print(f"Saved servers_at_risk.csv ({len(at_risk)} rows)")

> 💡 **Explanation**
>
> **Line by line**
>
> **Top of the file: the findings (step 9).** Two sentences as comments. You write these **at the end**, after you have seen the output, but they go at the top of the file so that anyone who opens it sees the result first.
>
> - `import pandas as pd`: brings in the pandas library with the short name `pd`. Everyone uses this name, so you should too.
> - `import matplotlib.pyplot as plt`: the charting tool, with the short name `plt`.
>
> **Step 1: load and look**
> - `df = pd.read_csv("servers.csv")`: reads the whole file in one go and makes a **DataFrame**. A DataFrame is a **table** with rows and columns, like a sheet in Excel. In Week 4 we wrote a loop and `split(",")` for this job; here it's one line. The first row of the file (the header) automatically becomes the column names.
> - `df.head()`: shows the **first** five rows. Always the first thing to do after loading: "was it read correctly?".
> - `df.dtypes`: the type of each column. `used` and `total` became `int64` (whole numbers), and `hostname` and `region` became `str` (pandas 2 writes `object`). If a column that should be a number has become `str`, something in the file is broken.
> - `len(df)`: the number of **rows** = the number of servers (10).
> - `len(df.columns)`: the number of **columns** (4). `df.columns.tolist()` gives the column names as a normal Python list.
> - An empty `print()` just prints a blank line so the output isn't crowded.
>
> **Step 2: a new column**
> - `df["percent"] = (df["used"] / df["total"]) * 100`: this is the most important idea of the week. `df["used"]` is a **whole column** (a Series), and `df["total"]` is a whole column too. When you divide them, pandas divides **row by row**: row 0 with row 0, row 1 with row 1, ... without any loop. This is called a **vectorised** operation. Writing `df["percent"] = ...` adds the new column to the table.
>
> **Step 3: the filter**
> - `df["percent"] >= 90`: this makes a column of `True`/`False`, one for each server. It's called a **boolean mask**.
> - `df[ ...mask... ]`: only the rows that are `True` are kept. The result is a smaller DataFrame.
> - `len(high_usage)`: how many rows are left? 5.
> - `high_usage[["hostname", "region", "percent"]]`: **two pairs of square brackets**! The outer brackets mean "select columns", and the inner brackets are a **list** of column names. With one pair of brackets you can only get one column.
> - **"at or above"** means `>=`, not `>`. A server at exactly 90% must be counted too.
>
> **Step 4: grouping**
> - `df.groupby("region")`: puts the rows into **groups** by region: one group for east, one for west, one for north.
> - `["percent"]`: takes only the `percent` column from each group.
> - `.mean()`: works out the average of each group. The result is a Series whose index is the region names.
> - `.round(1)`: one decimal place.
> - `.sort_values(ascending=False)`: biggest to smallest, so the first one is the answer.
>
> **Step 5: sorting**
> - `df.sort_values("percent", ascending=False)`: sorts the whole table by `percent` in **descending** order. The original `df` is not touched; it returns a **new** table. That's why we stored it in `ranked`.
> - `ranked.iloc[0]`: `iloc` means "by **position**". `[0]` means the first row of the **sorted** table. The result is a single row (a Series), and you can read its columns with `most_at_risk['hostname']`.
> - **Trap:** don't write `ranked.loc[0]`! `loc` works with the **index label**, and label 0 belongs to srv-01, not to the first server of the sorted table. After sorting, the labels have moved around, but they have not changed.
> - `:.0f` in an f-string means no decimal places (`120.0` → `120`).
>
> **Step 6: classifying with `apply` + `lambda`**
> - `lambda p: ...`: a one-line **function with no name**. Each time, `p` is one number from the `percent` column.
> - `"OVER LIMIT" if p >= 100 else "WARNING" if p >= 90 else "OK"`: a **chained one-line if**. Read it from the left: "if 100 or more → OVER LIMIT, otherwise if 90 or more → WARNING, otherwise → OK". **The order matters:** check the bigger condition (100) first, or 120 would also match `>= 90` and become WARNING.
> - `df["percent"].apply(...)`: runs that lambda on **each** value in the column and returns the results as a new column.
> - `df["status"].value_counts()`: how many times does each value appear? OK 5, OVER LIMIT 3, WARNING 2.
>
> **Step 7: the chart**
> - `status_counts.plot(kind="bar")`: pandas draws a bar chart by itself using matplotlib. The names (the index) go on the x-axis and the numbers become the heights of the bars.
> - `plt.title` / `plt.xlabel` / `plt.ylabel`: the title and the axis labels. A chart without labels is only worth half as much.
> - `plt.xticks(rotation=0)`: makes the labels under the bars straight (horizontal). By default pandas turns them 90 degrees.
> - `plt.tight_layout()`: tidies the margins so nothing gets cut off.
> - `plt.savefig("servers_per_status.png")`: saves the chart as an **image** next to your file.
> - `plt.show()`: opens the chart window. **The order matters:** first `savefig`, then `show`. After you close the `show` window, the chart is cleared, and if you call `savefig` after that, it saves a **blank white** image. (This is the same point as Topic 4, Section 4: in a `.py` file, if you only make the chart and neither save nor show it, you see nothing.)
>
> **Step 8: saving**
> - `df[df["status"] == "OVER LIMIT"]`: the same filter as step 3, this time on text.
> - `.to_csv("servers_at_risk.csv", index=False)`: writes the table to a new CSV. `index=False` means **don't** write the row numbers (0, 1, 2, ...). Without it, an extra column with no name would appear at the start of the file.
>
> **Output (the important parts):**
> ```
> Servers at or above 90%: 5
>   hostname region  percent
> 2   srv-03   west    120.0
> 3   srv-04   west     93.0
> 4   srv-05  north    101.0
> 7   srv-08  north    110.0
> 8   srv-09   east     95.0
>
> Average percent per region (highest first):
> region
> north    88.7
> west     81.0
> east     71.8
> Name: percent, dtype: float64
>
> Most at risk: srv-03 (west) at 120%
>
> status
> OK            5
> OVER LIMIT    3
> WARNING       2
> Name: count, dtype: int64
> ```
>
> **Hand check:**
> - north: (101 + 110 + 55) ÷ 3 = 266 ÷ 3 = 88.67 → `88.7` ✓
> - west: (120 + 93 + 30) ÷ 3 = 243 ÷ 3 = `81.0` ✓
> - east: (87 + 45 + 60 + 95) ÷ 4 = 287 ÷ 4 = 71.75 → `71.8` ✓
> - Because `total` is 100 everywhere, `percent` is exactly the same as `used`. But we wrote the code in a general way, so it still works if a server has a capacity of 200.
>
> **Steps 3 and 6 together:** "90% or more" (5) = WARNING (2) + OVER LIMIT (3). The totals match ✓
>
> **Traps**
> - `df[df["percent"] >= 90 and df["region"] == "west"]` gives the error `ValueError: The truth value of a Series is ambiguous`. To combine two conditions on columns, use `&` (and) and `|` (or), and put **each condition in brackets**: `df[(df["percent"] >= 90) & (df["region"] == "west")]`.
> - Type the column name exactly. `df["Percent"]` with a capital P gives a `KeyError`.
> - `df.sort_values(...)` on its own does **not** change `df`. Either store the result in a variable, or write `df = df.sort_values(...)`.
>
> **Try it yourself:** draw a second chart too: the number of servers per **region** (`df["region"].value_counts().plot(kind="bar")`), and save it with a different name.

## Survey Results Report (AI / Data Science)

**Data:** `survey_results.csv`, in this week's folder: 10 responses, 3 categories, a `score` column, and an `answered` column (`1` if they actually answered, `0` if that row is a no-response).

1. **Load and look.** `.head()`, `.dtypes`, `len()`.
2. **Measure the gap.** What percentage of rows have `answered == 0`? This is Week 1's `(part / total) * 100`, just counted with a filter instead of typed in by hand: `len(df[df["answered"] == 0]) / len(df) * 100`.
3. **Filter it clean.** Build a second DataFrame containing only the rows where `answered == 1`: everything from here on uses that one, not the original.
4. **Group it.** Average `score`, per `category`, on the cleaned data.
5. **Sort it.** Which category has the strongest average score?
6. **Classify it.** Add a `result` column with `.apply()` and a lambda: `"Pass"` if `score >= 60`, else `"Fail"`.
7. **Chart it.** A bar chart of pass/fail counts: `.value_counts().plot(kind="bar")`, saved with `plt.savefig(...)`.
8. **Save it.** Write the cleaned, classified DataFrame to `survey_results_clean.csv`.
9. **Write two sentences.** What's the actual finding: which category is strongest, and what fraction of responses had to be dropped as unanswered?

In [ ]:
# FINDINGS (two sentences):
# Category A is the strongest, with an average score of 81.5 on the answered rows
# (B 76.7, C 61.5), and 6 of the 7 real answers are a Pass.
# 3 of the 10 rows (30%) were no-responses and had to be dropped before any averaging.

import pandas as pd
import matplotlib.pyplot as plt

# 1. Load and look
df = pd.read_csv("survey_results.csv")
print(df.head())
print()
print(df.dtypes)
print(f"\nRows: {len(df)}")

# 2. Measure the gap: what percentage of rows were not answered?
unanswered = df[df["answered"] == 0]
gap_percent = len(unanswered) / len(df) * 100
print(f"\nUnanswered rows: {len(unanswered)} of {len(df)} ({gap_percent:.0f}%)")

# 3. Filter it clean: keep only answered rows (.copy() = a new, independent DataFrame)
clean = df[df["answered"] == 1].copy()
print(f"Rows kept: {len(clean)}")

# 4. Group it: average score per category, on the clean data
category_average = clean.groupby("category")["score"].mean().round(1)
print("\nAverage score per category (clean data):")
print(category_average)

# Why we cleaned first: the same average on the ORIGINAL data
wrong_average = df.groupby("category")["score"].mean().round(1)
print("\nFor comparison, WITHOUT cleaning (the zeros drag it down):")
print(wrong_average)

# 5. Sort it: which category is strongest?
ranked = category_average.sort_values(ascending=False)
print(f"\nStrongest category: {ranked.index[0]} (average {ranked.iloc[0]})")

# 6. Classify it: Pass / Fail with apply + lambda
clean["result"] = clean["score"].apply(lambda s: "Pass" if s >= 60 else "Fail")
print()
print(clean)
print()
print(clean["result"].value_counts())

# 7. Chart it: pass/fail counts
clean["result"].value_counts().plot(kind="bar")
plt.title("Survey results (answered only)")
plt.xlabel("Result")
plt.ylabel("Number of students")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("survey_pass_fail.png")
print("\nSaved survey_pass_fail.png")
plt.show()

# 8. Save it: the cleaned, classified DataFrame
clean.to_csv("survey_results_clean.csv", index=False)
print("Saved survey_results_clean.csv")

> 💡 **Explanation**
>
> **Line by line**
>
> **Step 1:** the usual opening lines: `read_csv`, `head()`, `dtypes`, `len()`. Here `student_id` and `category` are text (`str`), and `score` and `answered` are whole numbers (`int64`).
>
> **Step 2: measuring the gap**
> - `unanswered = df[df["answered"] == 0]`: a filter: only the rows that were not answered. **Two equals signs** (`==`) to compare. One (`=`) means "give a value", and here it would give a syntax error.
> - `len(unanswered) / len(df) * 100`: three ÷ ten × a hundred = 30%. This is exactly the formula the task gives; I just split it into two lines so it's easier to read.
> - `:.0f` means no decimals: `30.0` → `30`.
>
> **Step 3: the clean data**
> - `clean = df[df["answered"] == 1].copy()`: only the answered rows.
> - **Why `.copy()`?** Because in step 6 we add a new column (`result`) to `clean`. In pandas 2, if you change the result of a filter without `.copy()`, you get a yellow warning called `SettingWithCopyWarning`. It means: "I'm not sure if you are changing a copy or the original table". `.copy()` says clearly "this is a **new, independent** table". In pandas 3 this warning has been removed, but `.copy()` does no harm, and your code runs cleanly on both versions.
> - From here on, **everything** uses `clean`, not `df`. The task asks for exactly this.
>
> **Step 4: grouping**
> - `clean.groupby("category")["score"].mean().round(1)`: group by category, take the score column, average, one decimal place.
> - **Comparing with the dirty data:** I ran the same line on `df` (the original) too, so you can see with your own eyes why cleaning matters:
>
> | Category | Clean (`clean`) | Dirty (`df`) |
> |---|---|---|
> | A | 81.5 | 40.8 |
> | B | 76.7 | 76.7 |
> | C | 61.5 | 41.0 |
>
> Category A goes from "strongest" to "weakest"! Only because of two zeros that were never real scores. B didn't change because it had no unanswered rows.
>
> **Step 5: sorting**
> - `category_average.sort_values(ascending=False)`: this time we are sorting a **Series** (not a DataFrame), so we don't need a column name.
> - `ranked.index[0]`: the name (label) of the first item → `"A"`.
> - `ranked.iloc[0]`: the **value** of the first item → `81.5`.
>
> **Step 6: Pass / Fail**
> - `lambda s: "Pass" if s >= 60 else "Fail"`: returns one word for each score.
> - `clean["result"] = ...`: a new column on the **clean** data.
>
> **Step 7: the chart**
> - `clean["result"].value_counts()`: Pass 6, Fail 1.
> - `.plot(kind="bar")`, then the title, the labels, `savefig` and finally `show`. The same pattern as the previous project.
>
> **Step 8:** `clean.to_csv("survey_results_clean.csv", index=False)`. The task doesn't mention `index=False`, but without it an extra numbered column with no name appears in the file. Always add it, unless you really need the index.
>
> **Output (the important parts):**
> ```
> Unanswered rows: 3 of 10 (30%)
> Rows kept: 7
>
> Average score per category (clean data):
> category
> A    81.5
> B    76.7
> C    61.5
> Name: score, dtype: float64
>
> Strongest category: A (average 81.5)
>
> result
> Pass    6
> Fail    1
> Name: count, dtype: int64
> ```
>
> **Hand check:**
> - A: (72 + 91) ÷ 2 = 81.5 ✓
> - B: (88 + 65 + 77) ÷ 3 = 230 ÷ 3 = 76.67 → `76.7` ✓
> - C: (54 + 69) ÷ 2 = 61.5 ✓
> - The only Fail: `S007` with 54 ✓
>
> **⚠ A point about the analysis (for your finding sentences):** categories A and C each have only **2** real answers. An average of two numbers is not very reliable. One more person could change the whole result. In a real report you would write this too.
>
> **Traps**
> - If you leave out `.copy()` and run on pandas 2, you will see the `SettingWithCopyWarning` warning. It's not an error and the code runs, but learn to prevent it.
> - `df.groupby("category").mean()` (without choosing a column) gives a `TypeError`, because it tries to average the text column `student_id` too. Either choose the column (`["score"]`) or write `mean(numeric_only=True)`.
>
> **Try it yourself:** work out the Pass percentage for **each category** (hint: make a 1/0 column, then use `groupby("category")` and `mean()`).

## Login Attempts Report (Cyber Security)

**Data:** `login_attempts.csv`, in this week's folder: 10 login sessions, some IPs appearing more than once, an `attempts` and a `failed` count per row.

1. **Load and look.** `.head()`, `.dtypes`, `len()`.
2. **Ask a question with a filter.** Which rows had 3 or more failed attempts in a single session? Boolean filtering, same shape as Topic 2's `over_limit` example.
3. **Ask a question with a group.** Total `failed` attempts, per `ip`: remember some IPs show up in more than one row, so this is a real `groupby()`, not just reading one row off.
4. **Sort it.** Which single IP has the most total failed attempts across all its sessions?
5. **Flag it.** Add a `suspicious` column with `.apply()` and a lambda: `True` if that row's `failed >= 3`, else `False`.
6. **Chart it.** A bar chart of failed attempts per `ip` (group first, then `.plot(kind="bar")`), saved with `plt.savefig(...)`.
7. **Save it.** Write only the `suspicious == True` rows to `flagged_ips.csv`.
8. **Write two sentences.** Which IP would you escalate first, and why: total failures, or attempts in one sitting?

In [ ]:
# FINDINGS (two sentences):
# 10.0.0.12 and 10.0.0.5 are tied on total failures (8 each), but 10.0.0.12 did all 8
# in ONE session (8 of 9 attempts failed), so I would escalate it first.
# 10.0.0.5 comes next: it failed in both of its sessions (5, then 3 of 3), a repeat pattern.

import pandas as pd
import matplotlib.pyplot as plt

# 1. Load and look
df = pd.read_csv("login_attempts.csv")
print(df.head())
print()
print(df.dtypes)
print(f"\nRows (sessions): {len(df)}")
print(f"Different IPs: {df['ip'].nunique()}")

# 2. A question with a filter: sessions with 3 or more failed attempts
many_failures = df[df["failed"] >= 3]
print(f"\nSessions with 3+ failed attempts: {len(many_failures)}")
print(many_failures)

# 3. A question with a group: total failed attempts per IP
failed_per_ip = df.groupby("ip")["failed"].sum()

# 4. Sort it: which IP has the most failed attempts in total?
failed_per_ip = failed_per_ip.sort_values(ascending=False)
print("\nTotal failed attempts per IP (highest first):")
print(failed_per_ip)

top_total = failed_per_ip.max()
top_ips = failed_per_ip[failed_per_ip == top_total]
print(f"\nMost failed attempts in total: {top_total}, by {top_ips.index.tolist()}")

# 5. Flag it: suspicious column with apply + lambda
df["suspicious"] = df["failed"].apply(lambda f: True if f >= 3 else False)
print()
print(df)

# 6. Chart it: failed attempts per IP (group first, then plot)
failed_per_ip.plot(kind="bar")
plt.title("Total failed login attempts per IP")
plt.xlabel("IP address")
plt.ylabel("Failed attempts")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("failed_per_ip.png")
print("\nSaved failed_per_ip.png")
plt.show()

# 7. Save it: only the suspicious rows
flagged = df[df["suspicious"] == True]
flagged.to_csv("flagged_ips.csv", index=False)
print(f"Saved flagged_ips.csv ({len(flagged)} rows)")

> 💡 **Explanation**
>
> **Line by line**
>
> **Step 1**
> - `read_csv`, `head()`, `dtypes`, `len()`: as before. `ip` and `region` are text, and `attempts` and `failed` are whole numbers.
> - **Note:** even though the `ip` column is full of numbers and dots, it is read as `str` (text). That's correct, because `10.0.0.5` is not a mathematical number (it has several dots). It's a **label**.
> - `df['ip'].nunique()`: "number of unique", that is, how many **different** values are there? 10 rows but 8 IPs. So two IPs are repeated (`10.0.0.5` and `10.0.0.9`). This shows why step 3 really needs `groupby`. (`nunique` is not in the task. It's just an extra look.)
>
> **Step 2: the filter**
> - `df[df["failed"] >= 3]`: the same shape as `over_limit` in Topic 2: a boolean mask inside square brackets. It found 4 sessions.
>
> **Step 3: the group**
> - `df.groupby("ip")["failed"].sum()`: put the rows into groups by IP, take the `failed` column from each group and **add it up** (`sum`, not `mean`). For example, `10.0.0.5` has two rows: 5 + 3 = 8.
> - If you didn't group and only looked at the rows, you would think the most failures for `10.0.0.5` was 5. But its total is 8.
>
> **Step 4: sorting and ties**
> - `failed_per_ip.sort_values(ascending=False)`: biggest to smallest. We stored the result back in the same variable.
> - `failed_per_ip.max()`: the biggest number = 8.
> - `failed_per_ip[failed_per_ip == top_total]`: **a filter on a Series**: all the IPs that have exactly 8. This way we don't lose the tie.
> - `top_ips.index.tolist()`: the IP names as a list: `['10.0.0.12', '10.0.0.5']`.
> - **Why isn't `head(1)` enough?** Because when two values are equal, which one comes first depends on the details of the sort. Your answer would be one of the two, without you noticing that someone else has the same score.
>
> **Step 5: flagging**
> - `lambda f: True if f >= 3 else False`: exactly what the task asks. **A nice point:** `f >= 3` is already `True` or `False`! So `lambda f: f >= 3` does the same job. You can even do it without apply: `df["suspicious"] = df["failed"] >= 3`. But the exercise was to practise `apply` and `lambda`.
>
> **Step 6: the chart**
> - `failed_per_ip.plot(kind="bar")`: "group first, then plot". The grouped Series is already ready to plot: the IPs on the x-axis, and the total failures as the bar heights. Because we sorted it, the bars go from tall to short.
> - `plt.xticks(rotation=45)`: IPs are long strings. We tilt them 45 degrees so they don't overlap.
> - `tight_layout()`, `savefig()` and finally `show()`.
>
> **Step 7: saving**
> - `df[df["suspicious"] == True]`: exactly what the task wrote. Because the column itself is `True`/`False`, the shorter `df[df["suspicious"]]` does the same job.
> - `to_csv("flagged_ips.csv", index=False)`: 4 rows.
>
> **Output (the important parts):**
> ```
> Sessions with 3+ failed attempts: 4
>            ip region  attempts  failed
> 0    10.0.0.5   east         7       5
> 3   10.0.0.12  north         9       8
> 5    10.0.0.5   east         3       3
> 8  172.16.0.9   west         6       4
>
> Total failed attempts per IP (highest first):
> ip
> 10.0.0.12      8
> 10.0.0.5       8
> 172.16.0.9     4
> 192.168.1.7    2
> 192.168.1.1    1
> 10.0.0.9       1
> 10.0.0.20      0
> 172.16.0.3     0
> Name: failed, dtype: int64
>
> Most failed attempts in total: 8, by ['10.0.0.12', '10.0.0.5']
> ```
>
> **Hand check:** `10.0.0.5`: 5 + 3 = 8 ✓ , `10.0.0.9`: 0 + 1 = 1 ✓ , the total of all failures: 5+0+1+8+0+3+2+0+4+1 = 24, and the total of the second column of the output is also 8+8+4+2+1+1+0+0 = 24 ✓
>
> **Answer to question 8 (which one do we escalate first?)**
> Both have a total of 8, so "total" can't decide. A second measure helps:
> - `10.0.0.12`: **all 8 in one session**, out of 9 attempts. That means it typed wrong passwords one after another. This is the pattern of a **brute-force attack** (trying passwords one by one), so **this one first**.
> - `10.0.0.5`: failed in **both** of its sessions (5 out of 7, then 3 out of 3, that is, 100%!). The pattern is "it comes back and tries again". Second place.
>
> There isn't one single "right" answer. What matters is that you take your **reason** from the data.
>
> **Traps**
> - `groupby("ip")["failed"].mean()` instead of `sum()` gives the average per session (`10.0.0.5` would be 4), not the total. The question says "total".
> - `df.groupby("ip").sum()` without choosing a column also "adds up" the text column `region` (it joins the texts together, like `eastnorth`). Always say which numeric column you want.
>
> **Try it yourself:** make a column `fail_rate = failed / attempts * 100` and see which session has the highest failure **rate** (hint: one session has 100%).